**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Databases

Every experiment you run produces data that outlives the script that made it. This workshop is about *keeping* that data queryable: relational modeling, SQL, and talking to a database from Python — using a sensor-logging scenario you'd actually meet in a signals lab. Everything runs on `sqlite3` from Python's standard library: zero installation.

## 0. Introduction

Why not CSV files? They rot: no types, no relationships, no protection against half-written rows, and every question becomes a bespoke parsing script. A relational database gives you **structure** (tables with types), **integrity** (constraints, transactions), and a **query language** (SQL) that answers questions you hadn't thought of when you designed the file.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) — functions, dicts, `with` blocks.
- Nothing else: `sqlite3` ships with Python.

---
### 🕐 Session 1 of 3 — *The Relational Model & SQL Basics* (~35 min)
**Goal:** design tables with keys and constraints; insert and query with SELECT/WHERE/ORDER BY.
**Feeds into:** Session 2 (joins & aggregation).

---

## 2. Tables, Keys, Constraints

💡 **Intuition.** A table is a *typed spreadsheet with rules*. The **primary key** gives every row an identity; a **foreign key** is a pointer from one table's rows to another's — the relational version of the C pointer from [Intro to C](../../Intro_Programming/Intro_C.ipynb) §5, except the database *refuses* to let it dangle.

Our scenario: a lab logs signal recordings from multiple sensors. Two entities → two tables:

- `sensors` — one row per physical device.
- `recordings` — one row per capture, pointing at its sensor.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")          # RAM-only; use a filename to persist
db.execute("PRAGMA foreign_keys = ON")    # SQLite needs this opt-in!

db.executescript("""
CREATE TABLE sensors (
    sensor_id   INTEGER PRIMARY KEY,
    name        TEXT NOT NULL UNIQUE,
    kind        TEXT NOT NULL CHECK (kind IN ('accelerometer', 'microphone', 'ecg')),
    fs_hz       REAL NOT NULL CHECK (fs_hz > 0)
);
CREATE TABLE recordings (
    rec_id      INTEGER PRIMARY KEY,
    sensor_id   INTEGER NOT NULL REFERENCES sensors(sensor_id),
    started_at  TEXT NOT NULL,            -- ISO 8601 timestamp
    n_samples   INTEGER NOT NULL,
    rms         REAL                      -- a computed feature, maybe NULL until processed
);
""")
print("schema created")

In [ ]:

# YOUR CODE HERE


### 2.1. Constraints Earn Their Keep

Bad data is *rejected at the door* — a guarantee no CSV can make.

In [ ]:

# YOUR CODE HERE


### 2.2. SELECT: Asking Questions

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 2 of 3 — *Joins, Aggregation & Transactions* (~35 min)
**Goal:** combine tables with JOIN; summarize with GROUP BY; make multi-step changes atomic.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (databases from Python, files & analytics).

---

## 3. Joins & Aggregation

💡 **Intuition.** A **JOIN** re-follows the foreign-key pointers to reassemble the split entities: "for each recording, look up its sensor's row and staple them together." **GROUP BY** then folds rows sharing a key into one summary row — the SQL analogue of a NumPy `reduce` along an axis.

In [ ]:

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


Note `AVG(rms)` quietly *skipped* the NULL — aggregate functions ignore NULLs. Learn this before it surprises you in a paper's results table.

## 4. Transactions

💡 **Intuition.** A transaction makes several statements **all-or-nothing**. Classic example: moving a recording between sensors touches two rows; a crash between the two updates would corrupt the story. `BEGIN … COMMIT` means the database never shows the world a half-done state — and `ROLLBACK` is your undo.

In [ ]:
# The update was rolled back automatically:

# YOUR CODE HERE


---
### 🕐 Session 3 of 3 — *Databases from Python, Safely & at Scale* (~40 min)
**Goal:** parameterized queries, storing experiment results, and when to reach beyond SQLite.
**Builds on:** Session 2.

---

## 5. Python Patterns

### 5.1. Parameterized Queries — the Only Way

Never build SQL with f-strings: user input containing a quote breaks the query at best, *becomes* the query at worst (SQL injection). The `?` placeholder passes values out-of-band, so data can never be mistaken for code.

In [ ]:
# SAFE: value is bound as data — no row matches, nothing else happens

# YOUR CODE HERE


### 5.2. A Reusable Results Logger

The pattern worth stealing: every training run / parameter sweep in your research logs into a table, and analysis is a query away — compare with juggling 40 `results_final_v2 (copy).csv` files.

In [ ]:
# pretend sweep — in real life these come from your training loop

# YOUR CODE HERE


### 5.3. Beyond SQLite

| Need | Reach for |
|---|---|
| Many concurrent writers, network access, roles | **PostgreSQL** — same SQL, industrial engine |
| Analytics over millions of rows / Parquet files | **DuckDB** — SQLite's analytical twin |
| Long dense signal data | Store *arrays* in files (HDF5/Parquet), *metadata + paths* in SQL — don't put 48 kHz samples one-per-row |

The SQL you wrote today transfers to all of them nearly verbatim.

## 6. Conclusion

You modeled entities as tables, protected them with constraints, asked real questions with joins and aggregation, made changes atomic, and built the experiment-logging habit. That's 90% of the database craft a signals/ML researcher needs.

---
## Where next

- [Intro to Operating Systems](../README.md#workshop-1--introduction-to-operating-systems-available) — the files, processes, and locks a database is built from.
- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) — the NumPy side of the "arrays in files, metadata in SQL" pattern.